### Tutorial Followed: [Web Scraping with Python - Beautiful Soup Crash Course](https://www.youtube.com/watch?v=XVv6mJpFOb0&t=2026s) and [Advanced Web Scraping Tutorial! (w/ Python Beautiful Soup Library)](https://www.youtube.com/watch?v=DcI_AZqfZVc&t=6s)

#### Importing Dependencies:

In [129]:
from bs4 import BeautifulSoup
import requests
import time
import random
import csv

In [2]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
}

'''I use headers, to make my scraping look more legitimate and reduce risks of IP blocks etc.'''

'I use headers, to make my scraping look more legitimate and reduce risks of IP blocks etc.'

#### Cleaning Poem Text:

In [131]:
import re

def clean_poem_text(text):
    if not text or text == 'N/A':
        return 'N/A'

    lines = text.split('\n')
    cleaned_lines = []

    blacklist_phrases = [
        'Add to anthology',
        'Load audio player',
        'This poem is in the public domain.',
        'All rights reserved',
        'Published in the United States',
        'Published in Canada',
        'Copyright',
        'From ',
        '©',
        '×',
    ]

    for line in lines:
        line = line.strip()
        if not line:
            cleaned_lines.append('')
            continue

        # This skips blacklisted phrases
        if any(phrase.lower() in line.lower() for phrase in blacklist_phrases):
            continue

        # This skips lines that look like dates
        if re.match(r'^\d{4}\s?[–-]\s?\d{0,4}$', line):  # Matches "1901 – 1967" or "1901 –"
            continue

        # This skips lines that are just a year (e.g., "1901")
        if re.match(r'^\d{4}$', line):
            continue

        cleaned_lines.append(line)

    # This get rids of multiple blank lines
    cleaned_text = '\n'.join(cleaned_lines)
    cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)

    return cleaned_text.strip()


In [132]:
poems = []

#### Scraping Poems from: [poets.org](https://poets.org/)

In [133]:
url = 'https://poets.org/poems?page=0'
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'lxml')

In [134]:
rows = soup.find_all('tr')

In [135]:
for row in rows[:5]:
    try:
        poem_title = row.find('td', class_='views-field-title')
        poem_link = poem_title.find('a')
        title = poem_link.get_text(strip=True)
        link = 'https://poets.org' + poem_link['href']
        
        author_name = row.find('td', class_='views-field-field-author')
        author = author_name.get_text(strip=True)

        date_td = row.find('td', class_='views-field-field-date-published')
        date = date_td.get_text(strip=True)

        time.sleep(random.uniform(1.5, 3.0))

        poem_response = requests.get(link, headers=headers)
        poem_soup = BeautifulSoup(poem_response.text, 'lxml')

        poem_div = poem_soup.find('div', id="block-stanza-content")
        poem_text_raw = poem_div.get_text(separator='\n') if poem_div else 'N/A'
        poem_text_cleaned = clean_poem_text(poem_text_raw)

        poems.append({
            'title': title,
            'author': author,
            'date': date,
            'text': poem_text_cleaned
        })

        print(f"Got: {title} by {author} ({date})")

        time.sleep(random.uniform(2.0, 4.0))

    except Exception as e:
        print(f"Skipped a poem due to error: {e}")
        continue


Skipped a poem due to error: 'NoneType' object has no attribute 'find'
Got: A Line-storm Song by Robert Frost (1913)
Got: The Weary Blues by Langston Hughes (1926)
Got: Morning in the Burned House by Margaret Atwood (1995)
Got: On Living by Nâzim Hikmet (1994)


#### Saving Poems to CSV:

In [136]:
with open('sample_poems.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['title', 'author', 'date', 'text'])
    writer.writeheader()
    writer.writerows(poems)

print("Finished, saved to sample_poems.csv")

Finished! Data saved to sample1e_poeqms.csv
